# Raw Data Download — MAXIMA

Download raw X-ray diffraction (XRD) data files from the aimdl/datafiles API. This notebook shows how to query files by data type, download them in parallel, and organize them by sample (IGSN).

In [ ]:
import sys
from pathlib import Path
import requests

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from aimdl_examples import (
    get_girder_client,
    download_item_to_disk,
    paginate_datafiles,
    get_output_dir,
)

## Configuration

Select what data to download:

### Data Type

Choose one of:
- **`xrd_raw`** — Raw XRD detector images
- **`xrd_calibrant_raw`** — Raw calibrant images
- **`xrd_derived`** — Processed XRD data (scan images, intensity curves, CSVs)
- **`xrd_calibrant_derived`** — Processed calibrant data
- **`xrd_metadata`** — Metadata and logs

Most data types allow optional filename filtering (see next cell).

In [ ]:
DATA_TYPE = "xrd_raw"

### Filename Filter (optional)

Applies to: `xrd_raw`, `xrd_calibrant_raw`, `xrd_derived`, `xrd_calibrant_derived`. Leave blank to download all, or choose one:

**For raw types** (`xrd_raw`, `xrd_calibrant_raw`):
- `"master"` — Master files (summary metadata)
- Frame numbers like `"000001"`, `"000002"`, etc. — Individual detector frames

**For derived types** (`xrd_derived`, `xrd_calibrant_derived`):
- `"scan.png"` — Scan image
- `"xrd.csv"` — Intensity data as CSV
- `"xrd.png"` — XRD plot image
- `".tiff"` — TIFF files (derived only)

The filter matches any file containing this string in its name.

In [ ]:
FILENAME_FILTER = ""

### Output Directory

Files are organized by sample ID (IGSN) in subdirectories. Files from the same sample go together.

In [ ]:
from datetime import datetime

# Auto-generate a timestamped output folder to avoid conflicts between runs
TIMESTAMP = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

# To override with a custom name, uncomment and edit the line below:
# TIMESTAMP = "my_custom_folder_name"

In [ ]:
FILTERABLE_TYPES = ["xrd_raw", "xrd_calibrant_raw", "xrd_derived", "xrd_calibrant_derived"]

OUTPUT_DIR = f"./{TIMESTAMP}/{DATA_TYPE}"
if DATA_TYPE in FILTERABLE_TYPES and FILENAME_FILTER:
    OUTPUT_DIR = f"{OUTPUT_DIR}/{FILENAME_FILTER}"

## Download Files

This downloads all matching files in parallel using a thread pool. Files are prefixed with their item ID (first 8 characters) for uniqueness.

In [ ]:
with requests.Session() as session:
    gc = get_girder_client(session=session)
    
    # Set up filter if downloading with a filename filter
    item_filter = None
    if DATA_TYPE in FILTERABLE_TYPES and FILENAME_FILTER:
        def item_filter(item):
            return FILENAME_FILTER in item.get("name", "")
    
    # Download files
    results = paginate_datafiles(
        gc,
        DATA_TYPE,
        worker_fn=lambda item: download_item_to_disk(
            gc,
            {**item, "name": f"{item['_id'][:8]}_{item['name']}"},
            get_output_dir(item, OUTPUT_DIR)
        ),
        item_filter=item_filter
    )

In [ ]:
# Print summary
for result in results:
    print(f"Downloaded: {result['name']} ({result['size'] / 1024:.2f} KB)")
    
print(f"\nTotal files downloaded: {len(results)}")
print(f"Total size: {sum(r['size'] for r in results) / (1024**2):.2f} MB")

## Summary

View the downloaded files:

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
print(f"Downloaded {len(df)} files of type '{DATA_TYPE}'")
print(f"Output directory: {OUTPUT_DIR}")
print(f"\nSample files:")
print(df[["name", "size"]].head(10))